# Module 13 — Backtracking and Constraint Satisfaction

## What you will discover

Every cell below runs this module's **real** problem-bank solutions and asserts
their behaviour. Nothing here prints a claim it has not computed.

The assertions are lifted directly from `problems/tests/`, so they cannot drift
from the implementations — if a signature changes, the tests break first.

**One cell near the end is deliberately broken.** Fixing it is the exercise.

## Setup

The solutions directory goes on `sys.path` relative to this notebook's own
location. Never hard-code an absolute path — `tools/check_links.py` fails the
build on them, because a path with a username in it works on exactly one
machine.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "problems" / "solutions"))

from p01_subsets import subsets
from p02_permutations import permutations
from p03_subsets_with_dups import subsets_with_dups

print("module 13: Backtracking and Constraint Satisfaction")
print("problems available:", 8)
for name in ['p01_subsets', 'p02_permutations', 'p03_subsets_with_dups', 'p04_combination_sum', 'p05_generate_parentheses', 'p06_word_search', 'p07_n_queens', 'p08_palindrome_partition']:
    print(f"  {name}")

## 1. Baseline — `p01_subsets`

The first property, asserted rather than printed. Read the assertions before
running: each one names a specific input class, and most cross-check against an
independent brute force over the same data.

In [ ]:
assert subsets([1, 2, 3]) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
assert subsets([]) == [[]]
assert subsets([1]) == [[], [1]]
assert subsets([1, 2]) == [[], [1], [1, 2], [2]]
# There must be exactly 2^n subsets, all distinct.
for n in range(0, 9):
    got = subsets(list(range(n)))
    assert len(got) == 2 ** n, n
    assert len({tuple(x) for x in got}) == 2 ** n, n
# Every subset must be a real subset, and each element sorted.
data = [4, 1, 7]
got = subsets(data)
assert all(set(sub) <= set(data) for sub in got)
assert all(sub == sorted(sub) for sub in got)

print("all assertions held")

## 2. Predict before you run

Predict how many subsets `subsets([1, 2, 3])` returns, and predict what each of them contains if the code records `path` instead of `path[:]`. The count is right in both cases.

Commit to an answer before executing the next cell. Predicting and being wrong
is what makes the correction stick; reading the output first does not.

In [ ]:
assert permutations([1, 2, 3]) == [
    [1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1],
]
assert permutations([1]) == [[1]]
assert permutations([1, 2]) == [[1, 2], [2, 1]]
# There must be exactly n! permutations, all distinct.
import math
for n in range(1, 7):
    got = permutations(list(range(n)))
    assert len(got) == math.factorial(n), n
    assert len({tuple(x) for x in got}) == math.factorial(n), n
# Cross-check against itertools.
import itertools
for data in ([1, 2, 3], [5, 1, 9], [2, 4]):
    expected = sorted(list(p) for p in itertools.permutations(sorted(data)))
    assert permutations(data) == expected, data

print("all assertions held")

## 3. Measurement

Claims about complexity are claims about wall-clock behaviour at scale, so they
have to be measured rather than asserted from the shape of the code.

In [ ]:
started = time.perf_counter()

assert subsets_with_dups([1, 2, 2]) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
assert subsets_with_dups([]) == [[]]
assert subsets_with_dups([1, 1]) == [[], [1], [1, 1]]
# No duplicates: identical to the plain power set.
assert subsets_with_dups([1, 2, 3]) == subsets([1, 2, 3])
# All identical: n+1 subsets, one of each length.
assert subsets_with_dups([2, 2, 2]) == [[], [2], [2, 2], [2, 2, 2]]
# The results must be unique.
for data in ([1, 2, 2], [4, 4, 4, 1, 4], [1, 1, 2, 2], [0]):
    got = subsets_with_dups(data)
    assert len({tuple(x) for x in got}) == len(got), data
# Cross-check against dedup-after-the-fact.
import itertools
for data in ([1, 2, 2], [1, 1, 2, 2], [3, 3, 3]):
    expected = sorted(
        {tuple(sorted(c)) for r in range(len(data) + 1)
         for c in itertools.combinations(sorted(data), r)}
    )
    assert [tuple(x) for x in subsets_with_dups(data)] == expected, data

elapsed = (time.perf_counter() - started) * 1000
print(f"all assertions held in {elapsed:.2f} ms")

## 4. Fix this cell

The values below are **wrong on purpose**. Run it, read the failure, work out
the right numbers from the cells above, and correct them in place.

Change only the expected values — not the code that computes them.

In [ ]:
import os

# DELIBERATELY BROKEN - two expected values, both wrong. Fix in place.

expected_problem_count = 99      # how many problems does this module ship?
expected_solution_count = 99     # how many reference solutions are on disk?

problem_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems") if f.startswith("p") and f.endswith(".py")
)
solution_files = sorted(
    f for f in os.listdir(Path.cwd() / "problems" / "solutions")
    if f.startswith("p") and f.endswith(".py")
)

assert expected_problem_count == len(problem_files), (
    f"expected {expected_problem_count} problems, found {len(problem_files)}"
)
assert expected_solution_count == len(solution_files), (
    f"expected {expected_solution_count} solutions, found {len(solution_files)}"
)
print("Both match. Every problem has exactly one reference solution.")

## Takeaways

1. Record `path[:]`, not `path`. The list you are appending is about to be mutated.
2. Every change on the way down needs its undo on the way back up - all of them.
3. `i` versus `i + 1` in the recursive call is with- versus without-replacement.

### Where to go next

- [`01_README.md`](01_README.md) — the concepts in depth
- [`problems/README.md`](problems/README.md) — all 8 problems, with hint ladders
- [`debug_lab/SYMPTOMS.md`](debug_lab/SYMPTOMS.md) — planted defects that exit 0
- [Pattern Recognition Guide](../PATTERN_RECOGNITION_GUIDE.md) — attacking an unseen problem